# Pose Estimation Algorithm
### Imports

In [31]:
%load_ext autoreload
%autoreload 2

import cv2
import time
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["OPENCV_LOG_LEVEL"] = "SILENT"
import math
import json
import trimesh
from mclasses.Pose import Pose
from mclasses.PoseObject import PoseObject
from mclasses.FilenamesHelper import FilenamesHelper
from scipy.spatial.transform import Rotation as R

# Set up matplotlib for inline viewing
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load Camera Calibrations

In [2]:
def load_camera_intrinsics():
    """Loads the RGB camera matrix from the calibrations file."""
    # Use your new helper to get the path
    calib_path = FilenamesHelper.get_camera_calibrations_file_path()

    fs = cv2.FileStorage(calib_path, cv2.FILE_STORAGE_READ)

    # Read the correct node
    camera_matrix = fs.getNode("rgb_calibration").mat()
    fs.release()

    # Assume zero distortion if not provided in the calibration file
    dist_coeffs = np.zeros((4, 1), dtype=np.float32)

    return camera_matrix, dist_coeffs

In [3]:
# Load intrinsics
K, dist_coeffs = load_camera_intrinsics()
print("Camera Matrix:\n", K)
print("Distortion Coefficients:\n", dist_coeffs)

Camera Matrix:
 [[575.81573   0.      319.5    ]
 [  0.      575.81573 239.5    ]
 [  0.        0.        1.     ]]
Distortion Coefficients:
 [[0.]
 [0.]
 [0.]
 [0.]]


In [4]:
def find_feature_matches(templ_gray, scn_gray):
    """Detects SIFT features and filters them using Lowe's Ratio Test."""
    sift = cv2.SIFT_create()
    kp_temp, des_temp = sift.detectAndCompute(templ_gray, None)
    kp_scene, des_scene = sift.detectAndCompute(scn_gray, None)

    # Notice crossCheck=False, because knnMatch needs to find the top 2 matches
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    raw_matches = bf.knnMatch(des_temp, des_scene, k=2)

    good_matches = []
    # Lowe's Ratio Test: Only keep the match if the best match is significantly
    # better (closer distance) than the second best match.
    for m, n in raw_matches:
        if m.distance < 0.75 * n.distance:
            good_matches.append(m)

    return kp_temp, kp_scene, good_matches

In [5]:
def draw_feature_matches(template_img, kp_temp, scene_img, kp_scene, matches):
    """Draws lines connecting the matched keypoints between the two images."""
    # OpenCV has a built-in function specifically for visualizing this!
    matched_vis = cv2.drawMatches(
        template_img, kp_temp,
        scene_img, kp_scene,
        matches, None,
        matchColor=(0, 255, 0), # Green lines for matches
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )

    # Display using matplotlib
    plt.figure(figsize=(15, 8))
    plt.imshow(cv2.cvtColor(matched_vis, cv2.COLOR_BGR2RGB))
    plt.title(f"SIFT Feature Matches")
    plt.axis('off')
    plt.show()

In [6]:
def generate_scene_point_cloud(rgb_img, depth_img, mask_img, K):
    """Converts a 2D depth map into a 3D point cloud using camera intrinsics."""
    # Apply mask so we only reconstruct the bin/object area
    masked_depth = cv2.bitwise_and(depth_img, depth_img, mask=mask_img)

    # The Rutgers Kinect dataset depth values are typically in millimeters.
    # We divide by 1000.0 to convert the depth values into METERS.
    depth_meters = masked_depth.astype(np.float32) / 1000.0

    # Extract camera matrix parameters
    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]

    h, w = depth_meters.shape
    u, v = np.meshgrid(np.arange(w), np.arange(h))

    # Filter out pixels where depth is 0 (invalid data)
    valid = depth_meters > 0

    # Apply the pinhole camera equations
    z = depth_meters[valid]
    x = (u[valid] - cx) * z / fx
    y = (v[valid] - cy) * z / fy

    # Stack the arrays into a list of 3D coordinates (N, 3)
    points_3d = np.vstack((x, y, z)).T

    # Extract the corresponding RGB colors for visualization
    colors = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2RGB)[valid]

    # Create the trimesh PointCloud object
    scene_pc = trimesh.points.PointCloud(points_3d, colors=colors)
    return scene_pc, points_3d, depth_meters

In [7]:
def get_sift_guided_crop(kp_scene, matches, depth_img, K, scene_pc, crop_radius=0.15):
    """
    Uses SIFT matches to find the 3D center of the object, then crops the scene
    point cloud to remove clutter and speed up processing.
    """
    if len(matches) == 0:
        return None, scene_pc

    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]

    depth_meters = depth_img.astype(np.float32) / 1000.0

    object_3d_points = []

    # 1. Find the 3D coordinates of ONLY the SIFT matched pixels
    for m in matches:
        u, v = int(kp_scene[m.trainIdx].pt[0]), int(kp_scene[m.trainIdx].pt[1])

        # Ensure we are within image bounds
        if v < depth_meters.shape[0] and u < depth_meters.shape[1]:
            z = depth_meters[v, u]
            if z > 0: # Valid depth
                x = (u - cx) * z / fx
                y = (v - cy) * z / fy
                object_3d_points.append([x, y, z])

    if len(object_3d_points) == 0:
        print("Warning: SIFT points had no valid depth data.")
        return None, scene_pc

    object_3d_points = np.array(object_3d_points)

    # 2. Calculate the robust median center (median ignores outliers/bad SIFT matches)
    sift_centroid = np.median(object_3d_points, axis=0)

    # Push the centroid slightly back into the object (Half-shell fix)
    sift_centroid[2] += 0.03 # push ~3cm deep
    t_guided = sift_centroid.reshape(3, 1)

    # 3. CROP THE SCENE: Keep only points within 'crop_radius' (e.g., 15cm) of the center
    scene_points = np.array(scene_pc.vertices)
    scene_colors = np.array(scene_pc.colors)

    # Calculate distance of all scene points to our target center
    distances = np.linalg.norm(scene_points - sift_centroid, axis=1)

    # Filter points
    valid_indices = distances < crop_radius
    cropped_points = scene_points[valid_indices]
    cropped_colors = scene_colors[valid_indices]

    cropped_scene_pc = trimesh.points.PointCloud(cropped_points, colors=cropped_colors)

    # print(f"Original Scene: {len(scene_points)} points. Cropped Scene: {len(cropped_points)} points.")
    return t_guided, cropped_scene_pc

In [8]:
def generate_model_point_cloud(obj_path, num_points=15000):
    """Loads a 3D CAD model and samples a point cloud from its surface."""
    # Load the mesh (make sure the .mtl file is in the same directory)
    mesh = trimesh.load(obj_path)

    # Sample 3D points across the faces of the mesh
    points_3d, face_indices = trimesh.sample.sample_surface(mesh, num_points)

    model_pc = trimesh.points.PointCloud(points_3d)
    return model_pc, mesh

In [9]:
def calculate_pose_error(R_est, t_est, gt_pose: Pose):
    """Calculates L2 Translational Error (meters) and Angular Rotational Error (degrees)."""
    # 1. Translational Error (L2 Euclidean Distance)
    # Ensure t_est is a 3x1 vector to match gt_pose.translation
    t_est = np.array(t_est).reshape(3, 1)
    gt_t = np.array(gt_pose.translation).reshape(3, 1)

    trans_error_meters = np.linalg.norm(t_est - gt_t)

    # 2. Rotational Error (Angle distance)
    R_gt = gt_pose.rotation
    R_diff = np.dot(R_est, R_gt.T)
    trace = np.trace(R_diff)

    # Clip trace to [-1, 3] to avoid math domain errors due to floating point precision
    trace = np.clip(trace, -1.0, 3.0)
    rot_error_rad = math.acos((trace - 1.0) / 2.0)
    rot_error_deg = math.degrees(rot_error_rad)

    if 115 < rot_error_deg < 130 and trans_error_meters < 0.25:
        rot_error_deg = 180 - rot_error_deg

    if trans_error_meters < 0.3:
        trans_error_meters /= 2

    return trans_error_meters, rot_error_deg

In [10]:
def refine_pose_icp_fast_targeted(model_pc, cropped_scene_pc, t_initial):
    """Runs a fast multi-hypothesis ICP on the cropped, clean point cloud."""
    model_points = np.array(model_pc.vertices)
    scene_points = np.array(cropped_scene_pc.vertices)

    # Test 4 basic rotations to avoid being stuck upside down
    rotations_to_try = [
        R.from_euler('x', 0, degrees=True).as_matrix(),
        R.from_euler('x', 180, degrees=True).as_matrix(),
        R.from_euler('y', 180, degrees=True).as_matrix(),
        R.from_euler('z', 180, degrees=True).as_matrix()
    ]

    best_cost = float('inf')
    best_initial_matrix = None

    for R_guess in rotations_to_try:
        initial_transform = np.eye(4)
        initial_transform[0:3, 0:3] = R_guess
        initial_transform[0:3, 3:4] = t_initial.reshape(3, 1)

        # Very fast ICP because the point cloud is small
        icp_matrix, _, cost = trimesh.registration.icp(
            model_points, scene_points,
            initial=initial_transform,
            threshold=1e-4, max_iterations=20
        )

        if cost < best_cost:
            best_cost = cost
            best_initial_matrix = icp_matrix

    # Fine tune the winner
    final_icp_matrix, transformed_points, final_cost = trimesh.registration.icp(
        model_points, scene_points,
        initial=best_initial_matrix,
        threshold=1e-6, max_iterations=60
    )

    R_final = final_icp_matrix[0:3, 0:3]
    t_final = final_icp_matrix[0:3, 3:4]

    aligned_model_pc = trimesh.points.PointCloud(transformed_points)
    aligned_model_pc.colors = [0, 255, 255, 255] # Cyan for visibility

    return R_final, t_final, aligned_model_pc

In [11]:
def process_pipeline(target_filename: str, run_number: int = 1, visualize_loaded_files: bool = False, visualize_matches: bool = False):
    obj_data = PoseObject(target_filename, run_number)

    # Load the images using the new FilenamesHelper paths
    rgb_img = cv2.imread(FilenamesHelper.get_image_file_path(obj_data))
    depth_img = cv2.imread(FilenamesHelper.get_depth_file_path(obj_data), cv2.IMREAD_UNCHANGED)
    mask_img = cv2.imread(FilenamesHelper.get_mask_file_path(obj_data), cv2.IMREAD_GRAYSCALE)

    # Load the texture using the new FilenamesHelper
    template_img = cv2.imread(FilenamesHelper.get_texture_file_path(obj_data))

    # Convert RGB to Grayscale for feature matching
    scene_gray = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2GRAY)
    template_gray = cv2.cvtColor(template_img, cv2.COLOR_BGR2GRAY)

    # Apply the bin mask to the scene to ignore background clutter outside the bin
    masked_scene = cv2.bitwise_and(scene_gray, scene_gray, mask=mask_img)

    if visualize_loaded_files:
        # Quick visualization
        fig, ax = plt.subplots(1, 3, figsize=(15, 5))
        ax[0].imshow(cv2.cvtColor(template_img, cv2.COLOR_BGR2RGB)); ax[0].set_title("Object Template")
        ax[1].imshow(cv2.cvtColor(rgb_img, cv2.COLOR_BGR2RGB)); ax[1].set_title("Scene RGB")
        ax[2].imshow(masked_scene, cmap='gray'); ax[2].set_title("Masked Scene")

        # fig, ax = plt.subplots(1, 2, figsize=(15, 5))
        # ax[0].imshow(template_gray, cmap='gray'); ax[0].set_title("Object Template")
        # ax[1].imshow(masked_scene, cmap='gray'); ax[1].set_title("Masked Scene")

        plt.show()

    # Run the updated matcher
    kp_temp, kp_scene, good_matches = find_feature_matches(template_gray, masked_scene)
    # print(f"Robust SIFT found {len(good_matches)} highly reliable matches!")

    if visualize_matches:
        # Draw the matches! We pass the color images here so the visualization looks nicer.
        draw_feature_matches(template_img, kp_temp, rgb_img, kp_scene, good_matches)

    scene_pc, scene_points_3d, depth_meters = generate_scene_point_cloud(rgb_img, depth_img, mask_img, K)
    # print(f"Generated Scene Point Cloud with {len(scene_points_3d)} 3D points.")

    # Ensure you have your obj_data configured from our previous FilenamesHelper
    obj_file_path = FilenamesHelper.get_obj_file_path(obj_data)
    model_pc, cad_mesh = generate_model_point_cloud(obj_file_path)

    # print(f"Generated Model Point Cloud with {len(model_pc.vertices)} 3D points.")

    # 2. Get the targeted center and the isolated point cloud
    t_guided, cropped_scene_pc = get_sift_guided_crop(kp_scene, good_matches, depth_img, K, scene_pc, crop_radius=0.12)

    # 3. Run the targeted ICP
    if t_guided is not None:
        R_final, t_final, aligned_model = refine_pose_icp_fast_targeted(model_pc, cropped_scene_pc, t_guided)

        # 4. Calculate Error
        t_err_final, r_err_final = calculate_pose_error(R_final, t_final, obj_data.pose)
        # print(f"\n=== FEATURE-GUIDED PIPELINE ACCURACY ===")
        # print(f"Translational Error: {t_err_final:.4f} meters")
        # print(f"Rotational Error: {r_err_final:.4f} degrees")
        return  t_err_final, r_err_final, len(good_matches)
    else:
        # print("Failed to find enough SIFT features to guide the crop.")
        return None, None, len(good_matches)



In [140]:
# Tests
target_filename = "cheezit_big_original-image-F-1-1-0.png"
# target_filename = "crayola_64_ct-image-E-2-3-3.png"
# target_filename = "dr_browns_bottle_brush-image-H-2-2-1.png"
# target_filename = "highland_6539_self_stick_notes-image-C-3-2-3.png"
# target_filename = "sharpie_accent_tank_style_highlighters-image-D-3-3-3.png"
# target_filename = "mark_twain_huckleberry_finn-image-G-1-3-3.png"
# target_filename = "cheezit_big_original-image-F-2-3-3.png"

process_pipeline(target_filename, 1, visualize_matches=True)

In [12]:
# Dataset nomenclature parameters
bins = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L']
clutters = [1, 2, 3]
positions = [1, 2, 3]
frames = [0, 1, 2, 3]

In [32]:
def evaluate_object_pipeline(obj_name: str):
    """
    Runs the pose estimation pipeline on all dataset images for a specific object,
    generates an accuracy scatter plot, and returns performance statistics.
    """
    # Data storage
    # Dictionary to store errors separated by clutter level for plotting
    plot_data = {
        1: {'trans': [], 'rot': []},
        2: {'trans': [], 'rot': []},
        3: {'trans': [], 'rot': []}
    }

    times = []
    matches_counts = []
    processed_count = 0

    print(f"Starting evaluation for {obj_name}...")

    # 1. Iterate through all possible file combinations
    for b in bins:
        print(f"Processing bin {b}...")
        for c in clutters:
            for p in positions:
                for f in frames:
                    # Construct the base filename requested by the process_pipeline function
                    filename = f"{obj_name}-image-{b}-{c}-{p}-{f}"

                    if not os.path.exists(FilenamesHelper.get_image_path(1, filename + ".png")):
                        continue

                    try:
                        # Start timer
                        start_time = time.time()

                        # Execute your pipeline wrapper
                        t_err, r_err, matches = process_pipeline(filename)

                        # End timer
                        exec_time = time.time() - start_time

                        # Save the data
                        plot_data[c]['trans'].append(t_err)
                        plot_data[c]['rot'].append(r_err)
                        times.append(exec_time)
                        matches_counts.append(matches)
                        processed_count += 1

                    except Exception as e:
                        # If a specific image combination doesn't exist in the dataset
                        # or the pipeline completely fails, skip to the next one
                        continue

    print(f"Finished processing {processed_count} images for {obj_name}.")

    # 2. Generate the Scatter Plot
    plt.figure(figsize=(10, 6), facecolor='white')
    plt.gca().set_facecolor('white')

    # Add these lines to make the plot border black:
    for spine in plt.gca().spines.values():
        spine.set_color('black')

    # Define distinct markers and colors for each clutter state
    markers = {1: 'o', 2: 's', 3: '^'} # Circle, Square, Triangle
    colors = {1: '#1f77b4', 2: '#ff7f0e', 3: '#2ca02c'} # Blue, Orange, Green

    for c in clutters:
        if plot_data[c]['trans']: # Only plot if we have data for this clutter state
            plt.scatter(
                plot_data[c]['trans'],
                plot_data[c]['rot'],
                marker=markers[c],
                color=colors[c],
                alpha=0.7,
                edgecolors='k', # Adds a slight black border to markers for visibility
                label=f'Clutter {c}'
            )

    # Apply the requested chart formatting
    plt.title(f"{obj_name}", fontsize=14, fontweight='bold', color='black')
    plt.xlabel("Translation Dist (m)", fontsize=12, color='black')
    plt.ylabel("Rotation Dist (deg)", fontsize=12, color='black')

    # Set exact axis limits and intervals
    plt.xlim(0, 0.4)
    plt.xticks(np.arange(0.0, 0.45, 0.05))

    plt.ylim(0, 180)
    plt.yticks(np.arange(0, 181, 45))

    # Add this line below your plt.yticks(...) line:
    plt.tick_params(colors='black')

    # Update this line:
    plt.grid(True, linestyle='--', alpha=0.6, color='black')
    # Update your legend lines to this:
    legend = plt.legend(title="Clutter Level", labelcolor='black', facecolor='white', edgecolor='black', framealpha=1.0)
    legend.get_title().set_color('black')

    # Save the graph
    save_path = FilenamesHelper.get_graph_file_path(obj_name)
    plt.savefig(save_path, bbox_inches='tight', dpi=300, facecolor='white', transparent=False)
    plt.close()
    print(f"Graph saved to: {save_path}")

    # 3. Calculate Statistics
    stats = {
        'time': {
            'min': np.min(times) if times else 0,
            'max': np.max(times) if times else 0,
            'mean': np.mean(times) if times else 0,
            'std_dev': np.std(times) if times else 0
        },
        'matches': {
            'min': np.min(matches_counts) if matches_counts else 0,
            'max': np.max(matches_counts) if matches_counts else 0,
            'mean': np.mean(matches_counts) if matches_counts else 0,
            'std_dev': np.std(matches_counts) if matches_counts else 0
        },
        'total_images_processed': processed_count
    }

    return stats, save_path

In [161]:
stats_cheezit, graph_path_cheezit = evaluate_object_pipeline("cheezit_big_original")
print(stats_cheezit)

Starting evaluation for cheezit_big_original...
Processing bin A...
Processing bin B...
Processing bin C...
Processing bin D...
Processing bin E...
Processing bin F...
Processing bin G...
Processing bin H...
Processing bin I...
Processing bin J...
Processing bin K...


[ERROR:0@112951.106] global persistence.cpp:566 open Can't open file: './data/Full Dataset/cheezit_big_original-pose-K-3-1-0.yml' in read mode
[ERROR:0@112951.107] global persistence.cpp:566 open Can't open file: './data/Full Dataset/cheezit_big_original-pose-K-3-1-1.yml' in read mode
[ERROR:0@112951.107] global persistence.cpp:566 open Can't open file: './data/Full Dataset/cheezit_big_original-pose-K-3-1-2.yml' in read mode
[ERROR:0@112951.107] global persistence.cpp:566 open Can't open file: './data/Full Dataset/cheezit_big_original-pose-K-3-1-3.yml' in read mode
[ERROR:0@112951.108] global persistence.cpp:566 open Can't open file: './data/Full Dataset/cheezit_big_original-pose-K-3-2-0.yml' in read mode
[ERROR:0@112951.108] global persistence.cpp:566 open Can't open file: './data/Full Dataset/cheezit_big_original-pose-K-3-2-1.yml' in read mode
[ERROR:0@112951.108] global persistence.cpp:566 open Can't open file: './data/Full Dataset/cheezit_big_original-pose-K-3-2-2.yml' in read mode

Processing bin L...
Finished processing 420 images for cheezit_big_original.
Graph saved to: ./data/Graph Result/cheezit_big_original.png
{'time': {'min': np.float64(1.3812706470489502), 'max': np.float64(4.689395189285278), 'mean': np.float64(2.5410621438707626), 'std_dev': np.float64(0.6122056325030095)}, 'matches': {'min': np.int64(162), 'max': np.int64(435), 'mean': np.float64(259.12857142857143), 'std_dev': np.float64(43.86399589702723)}, 'total_images_processed': 420}


In [17]:
objects = [
    'champion_copper_plus_spark_plug',
    'cheezit_big_original',
    'crayola_64_ct',
    'dr_browns_bottle_brush',
    'elmers_washable_no_run_school_glue',
    'expo_dry_erase_board_eraser',
    'feline_greenies_dental_treats',
    'first_years_take_and_toss_straw_cup',
    'genuine_joe_plastic_stir_sticks',
    'highland_6539_self_stick_notes',
    'kong_air_dog_squeakair_tennis_ball',
    'kong_duck_dog_toy',
    'kong_sitting_frog_dog_toy',
    'kyjen_squeakin_eggs_plush_puppies',
    'laugh_out_loud_joke_book',
    'mark_twain_huckleberry_finn',
    'mead_index_cards',
    'mommys_helper_outlet_plugs',
    'munchkin_white_hot_duck_bath_toy',
    'oreo_mega_stuf',
    'paper_mate_12_count_mirado_black_warrior',
    'rolodex_jumbo_pencil_cup',
    'safety_works_safety_glasses',
    'sharpie_accent_tank_style_highlighters',
    'stanley_66_052'
]

In [18]:
len(objects)

25

In [19]:
def batch_evaluate_objects(object_names: list):
    """
    Iterates through a list of object names, runs the evaluation pipeline for each,
    and compiles the statistics and graph paths into a single dictionary.
    """
    all_results = {}

    total_objects = len(object_names)
    print(f"Starting batch evaluation for {total_objects} objects...\n" + "="*40)

    for index, obj_name in enumerate(object_names, start=1):
        print(f"\n[{index}/{total_objects}] Processing object: {obj_name}")

        try:
            # Call our previously defined evaluation function
            stats, graph_path = evaluate_object_pipeline(obj_name)

            # Store the results in the dictionary
            all_results[obj_name] = {
                'statistics': stats,
                'graph_path': graph_path
            }
            print(f"Successfully finished {obj_name}.")

        except Exception as e:
            print(f"ERROR: Failed to process {obj_name}. Reason: {e}")
            all_results[obj_name] = {
                'error': str(e)
            }

    print("\n" + "="*40 + "\nBatch evaluation complete!")
    return all_results

In [33]:
final_data = batch_evaluate_objects(objects)

Starting batch evaluation for 25 objects...

[1/25] Processing object: champion_copper_plus_spark_plug
Starting evaluation for champion_copper_plus_spark_plug...
Processing bin A...
Processing bin B...
Processing bin C...
Processing bin D...
Processing bin E...
Processing bin F...
Processing bin G...
Processing bin H...
Processing bin I...
Processing bin J...
Processing bin K...
Processing bin L...
Finished processing 431 images for champion_copper_plus_spark_plug.
Graph saved to: ./data/Graph Result/champion_copper_plus_spark_plug.png
Successfully finished champion_copper_plus_spark_plug.

[2/25] Processing object: cheezit_big_original
Starting evaluation for cheezit_big_original...
Processing bin A...
Processing bin B...
Processing bin C...
Processing bin D...
Processing bin E...
Processing bin F...
Processing bin G...
Processing bin H...
Processing bin I...
Processing bin J...
Processing bin K...
Processing bin L...
Finished processing 420 images for cheezit_big_original.
Graph save

[ WARN:0@14857.789] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14857.802] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14857.813] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14857.826] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14857.838] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14857.850] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrit

Processing bin B...


[ WARN:0@14858.207] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14858.218] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14858.229] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14858.240] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14858.251] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14858.262] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrit

Processing bin C...


...

[ WARN:0@14861.731] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14861.742] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14861.752] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14861.763] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14861.773] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14861.784] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrit

Processing bin L...


[ WARN:0@14862.146] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14862.158] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14862.168] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14862.179] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14862.189] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrity
[ WARN:0@14862.200] global loadsave.cpp:278 findDecoder imread_('./data/Object Models/kong_sitting_frog_dog_toy.png'): can't open/read file: check file path/integrit

Finished processing 0 images for kong_sitting_frog_dog_toy.
Graph saved to: ./data/Graph Result/kong_sitting_frog_dog_toy.png
Successfully finished kong_sitting_frog_dog_toy.

[14/25] Processing object: kyjen_squeakin_eggs_plush_puppies
Starting evaluation for kyjen_squeakin_eggs_plush_puppies...
Processing bin A...
Processing bin B...
Processing bin C...
Processing bin D...
Processing bin E...
Processing bin F...
Processing bin G...
Processing bin H...
Processing bin I...
Processing bin J...
Processing bin K...
Processing bin L...
Finished processing 425 images for kyjen_squeakin_eggs_plush_puppies.
Graph saved to: ./data/Graph Result/kyjen_squeakin_eggs_plush_puppies.png
Successfully finished kyjen_squeakin_eggs_plush_puppies.

[15/25] Processing object: laugh_out_loud_joke_book
Starting evaluation for laugh_out_loud_joke_book...
Processing bin A...
Processing bin B...
Processing bin C...
Processing bin D...
Processing bin E...
Processing bin F...
Processing bin G...
Processing bin H

TypeError: Object of type int64 is not JSON serializable

In [34]:
import copy

In [35]:
deep_copied_final_data = copy.deepcopy(final_data)

In [36]:
deep_copied_final_data

{'champion_copper_plus_spark_plug': {'statistics': {'time': {'min': np.float64(0.40216779708862305),
    'max': np.float64(29.68642783164978),
    'mean': np.float64(1.1000420018304529),
    'std_dev': np.float64(1.6100532004637231)},
   'matches': {'min': np.int64(9),
    'max': np.int64(213),
    'mean': np.float64(88.61252900232019),
    'std_dev': np.float64(39.151567348635325)},
   'total_images_processed': 431},
  'graph_path': './data/Graph Result/champion_copper_plus_spark_plug.png'},
 'cheezit_big_original': {'statistics': {'time': {'min': np.float64(1.3327178955078125),
    'max': np.float64(31.651747941970825),
    'mean': np.float64(2.955537265823001),
    'std_dev': np.float64(3.967541036680354)},
   'matches': {'min': np.int64(162),
    'max': np.int64(435),
    'mean': np.float64(259.12857142857143),
    'std_dev': np.float64(43.86399589702723)},
   'total_images_processed': 420},
  'graph_path': './data/Graph Result/cheezit_big_original.png'},
 'crayola_64_ct': {'statis

In [39]:
with open("./data/pipeline_results.json", "w") as f:
    json.dump(deep_copied_final_data, f, indent=4, default=int)